# 04 Inventory Performance Analysis — 開發日誌

**資料來源：**
- Kaggle: kaggle.com/bhanupratapbiswas/inventory-analysis-case-study
- PwC 是 PricewaterhouseCoopers 的縮寫，全球四大會計師事務所（Big Four）之一
- 數據集的業務邏輯設計符合真實審計和財務分析標準，欄位設計反映的是真實企業的採購流程


---
## 📅 2026-04-14 — Phase 1：Raw Data 載入

### ✅ 完成事項
- 建立 `raw` / `staging` / `marts` 三層 Schema
- 執行 `sql/00_schema_setup.sql` 成功建立所有 raw tables
- 完成 `scripts/01_load_raw.py` Python Loader 腳本

---

### 🪲 踩坑紀錄 1：`00_schema_setup.sql` 出現 NOTICE 訊息

**現象：**
```
NOTICE: table "raw_sales" does not exist, skipping
Successfully run. Total query runtime: 113 msec.
1 rows affected.
```

**原因：**
- `NOTICE` 只是 PostgreSQL 的溫馨提示，不是錯誤（ERROR）
- SQL 中使用了 `DROP TABLE IF EXISTS`，第一次執行時找不到資料表，PostgreSQL 會提示 skipping
- `1 rows affected` 是最後一行 `SELECT 'Schema setup complete ✅'` 回傳了 1 行結果

**結論：** ✅ 完全正常，可放心繼續執行

---

### 🪲 踩坑紀錄 2：執行 Python 腳本時彈出新視窗、自動關閉

**現象：**
- 在 VSCode PowerShell 執行 `01_load_raw.py` 時，自動跳出一個新的黑色 Console 視窗
- 視窗跑完後自動關閉，看不到輸出結果

**原因：**
- Windows 環境下，某些情況下 Python 腳本會以獨立視窗模式啟動
- 程式執行結果無法保留在 VSCode 終端機中

**解決方法：**
- 直接在 VSCode 的終端機（PowerShell）輸入指令執行，而不是按 Run 按鈕
- 加上 `-u` 參數（Unbuffered）：
```bash
python -u scripts/01_load_raw.py
```

---

### 🪲 踩坑紀錄 3：`KeyboardInterrupt` 錯誤反覆出現

**現象：**
```
Traceback (most recent call last):
  File "scripts/01_load_raw.py", line 12, in <module>
    from sqlalchemy import create_engine, text
  ...
KeyboardInterrupt
```

**原因：**
- 這不是程式本身的 Bug！
- Python 在 Windows 上第一次載入 `pandas`、`sqlalchemy` 等大型套件時，需要花 **30 秒至 1 分鐘** 從硬碟讀取套件檔案
- 期間畫面黑黑的沒有任何反應，誤以為程式當機（Hang），於是按下 `Ctrl+C` 強制中止
- `Ctrl+C` 就會產生 `KeyboardInterrupt` 錯誤

**解決方法：**
- 執行指令後，把手離開鍵盤，**耐心等待**
- 等到畫面出現以下文字才代表成功啟動：
```
=======================================================
04_Inventory_Performance_Analysis — Raw Loader
=======================================================
```


---

### 📖 今日學習筆記

| 知識點 | 說明 |
|---|---|
| `python -u` | Unbuffered 模式，強制即時輸出 print 內容，不等暫存區滿才印出 |
| `DROP TABLE IF EXISTS` | PostgreSQL 找不到資料表時不報錯，只顯示 NOTICE |
| `load_dotenv()` | 從 `.env` 檔案讀取環境變數（資料庫密碼），避免密碼寫在程式碼中 |
| `os.getenv('KEY', 'fallback')` | 讀取環境變數，若找不到則使用預設值 |
| `encoding='utf-8-sig'` | 處理 CSV 的 BOM（\ufeff）問題，避免第一個欄位名稱被污染 |
| `dtype=str` | 讀取 CSV 時全部當作字串，避免 Pandas 自動型別推斷出錯 |
| 虛擬環境 (`04_env`) | 隔離專案套件，不影響電腦其他 Python 環境；用 `deactivate` 退出 |


---

### 🪲 踩坑紀錄 4：終端機關掉後，如何回看 Raw Loader 執行結果

**問題：**
- `01_load_raw.py` 執行完成後，終端機 / Console 視窗已經關掉
- 無法直接複製當時印出的 row count 給專案紀錄

**解法：**
- 本專案的 Python loader 已經設計了 `raw.load_audit` 稽核表
- 每次載入成功後，會把：
  - `table_name`
  - `rows_loaded`
  - `loaded_at`
  - `source_file`
  寫進 PostgreSQL

**回看載入結果的 SQL：**
```sql
SELECT 
    table_name, 
    rows_loaded, 
    loaded_at, 
    source_file 
FROM raw.load_audit
ORDER BY loaded_at DESC;
```

**用途：**
- 就算終端機關掉，也能從資料庫重新查回本次載入結果
- 適合拿來做：
  - row count 驗證
  - load 成功證明
  - README / 開發日誌補寫
  - 後續 staging 前的資料核對

**心得：**
- 長時間執行的資料載入流程，不要只依賴終端機畫面
- 最好把結果落地到 audit table，之後查詢會比找截圖更可靠



***

## Phase 2 驗證結果總覽

### ✅ 完全通過的檢查

| 檢查項目 | 結果 |
|---|---|
| `raw_sales` 負數 qty / dollars | **0** — 乾淨 |
| `raw_sales` price calc mismatch | **0** — SalesDollars = Qty × Price 完全吻合 |
| `raw_purchases` dollar calc mismatch | **0** — 採購金額計算正確 |
| `raw_beg/end_inventory` 負庫存 | **0** — 無異常 |
| Purchases vendor → Invoice vendor | **0** — 外鍵完整 |

***

### ⚠️ 需要處理的 3 個問題

**問題 1：`raw_purchase_prices` — price vs purchase_price 差異 12,260 筆** 

這是最重要的發現。兩欄並存且大量不一致，業務含義推測：
- `price` = **零售售價（Retail Price）**
- `purchase_price` = **進貨成本（Cost Price）**

這兩個欄位在 staging 層必須用明確的業務名稱區分，是計算 **Gross Margin** 和 **Inventory Turnover** 的關鍵。

**問題 2：`raw_purchase_prices` — price 欄位有 2 筆 zero/null** 

數量少，但需要在 staging 層用 `NULLIF` 處理，避免除零錯誤影響 DSI 計算。

**問題 3：Sales → Inventory 孤兒記錄大量存在** 
- Sales 中有 **64,044 個** `inventory_id` 在 `raw_beg_inventory` 找不到
- Sales 中有 **50,368 個** `inventory_id` 在 `raw_end_inventory` 找不到

這**不是錯誤**，是業務正常現象——有些商品在期初/期末已售罄，庫存表只記錄有在手庫存的品項。Staging 層用 `LEFT JOIN` 處理即可。

***

### 5a / 5b 無結果 = 好消息

**完全沒有重複記錄** ——`raw_sales` 和 `raw_purchase_prices` 的自然鍵均唯一，pgAdmin 顯示空結果是正確的行為（`HAVING COUNT(*) > 1` 沒有符合條件的行）。

***

## Phase 2 驗證結論

```
整體資料品質：良好 ✅
核心數值計算：完全準確 ✅
最大風險點：purchase_prices 雙 price 欄位語意不清 ⚠️
```









***

## 專案開發紀錄：Phase 2 ~ Phase 4 (Data Validation to Star Schema)
**日期**：2026-04-15
**階段目標**：將 1500 萬筆 Raw CSV 資料進行驗證、清洗、型別轉換，並最終建立適合 Power BI 分析的星型結構 (Star Schema)。

### 📍 Phase 2: 原始資料驗證 (Raw Data Validation)
**目標**：透過 SQL 檢驗載入 `raw` schema 的 6 張表，確認資料品質（Data Quality）、空值、業務邏輯與關聯完整性。


*   **業務邏輯發現**：
    *   `raw_purchase_prices` 表中同時存在 `price` 與 `purchase_price`，且有 12,260 筆不一致。經分析確立業務含義：`price` 為終端零售價 (Retail Price)，`purchase_price` 為進貨成本 (Cost Price)。
    *   部分 `vendor_name` 存在尾部空白字元（Trailing whitespace），需在 Staging 層清理。

### 📍 Phase 3: 暫存層轉換與清洗 (Staging Layer)
**目標**：建立 `stg_` 表，將所有 `TEXT` 型別轉換為正確的數值與日期型別，並統一命名規範與清理髒資料。

*   **問題 1：整數轉型失敗 (包含小數點的數量)**
    *   **狀況**：報錯 `invalid input syntax for type integer: "162.5"`。
    *   **解決**：真實世界的庫存數據（如秤重商品或容量）帶有小數。將 `quantity`, `sales_quantity`, `on_hand`, `volume` 等欄位從 `INTEGER` 改為 `NUMERIC(10,2)`。
*   **問題 2：字串混入數值欄位 (Dirty Data)**
    *   **狀況**：報錯 `invalid input syntax for type numeric: "Unknown"`。
    *   **解決**：在 1500 萬筆資料中混入了 `"Unknown"` 這樣的無效字串。實作了**防禦性轉型 (Defensive Casting)**，使用 `NULLIF(NULLIF(TRIM(column), ''), 'Unknown')` 將預期外的字串安全轉換為 `NULL`，確保 Data Pipeline 的穩健性。    
*   **根據驗證結果，Staging 層需要處理以下 4 件事：

1. **`raw_purchase_prices`** — 將 `price` 重命名為 `retail_price`，`purchase_price` 保留為 `purchase_price`，語意明確化
2. **所有 TEXT 欄位** — CAST 成正確類型（`NUMERIC`, `DATE`, `INTEGER`）
3. **VendorName TRIM** — 去除尾部空格（Section 7 已確認存在）
4. **`vendor_no` → `vendor_number`** — 統一 `raw_sales` 的命名與其他表一致

### 📍 Phase 4: 資料超市層建模 (Marts Layer - Star Schema)
**目標**：根據 Kimball 维度建模理論，建立事實表與維度表，並引入代理鍵 (Surrogate Keys, SK) 提升後續 BI 工具的關聯效能。

*   **建模產出**：
    *   **維度表 (Dimensions)**：`dim_product` (合併售價與成本基準), `dim_vendor`, `dim_store`。
    *   **事實表 (Facts)**：`fact_sales` (交易型), `fact_inventory_snapshot` (週期快照型，結合期初與期末)。
*   **問題 ：SQL 語法嚴格性 (Ambiguous Column & GROUP BY)**
    *   **狀況**：建立 `dim_product` 時，發生 `column reference "brand" is ambiguous` 及 `must appear in the GROUP BY clause` 錯誤。
    *   **解決**：PostgreSQL 對 Window Function `ROW_NUMBER() OVER(ORDER BY ...)` 內的表達式要求極高，必須與 `GROUP BY` 完全一致。透過明確指定別名 (Alias) 如 `COALESCE(p.brand, s.brand)` 並預先對子查詢進行聚合 (Pre-aggregation) 來優化 `FULL OUTER JOIN` 效能，最終成功建立維度表。

***

## Phase 5 



## 1. 資料連接策略：Import vs DirectQuery


**PostgreSQL 端的建議預處理（在 Power BI 連接前）：**


這個 `dim_date` 的意義在於：DAX 的 Time Intelligence 函數（`DATEADD`、`TOTALYTD` 等）**必須**依賴一張連續無缺漏的日期表，且該表須標記為 Date Table。直接從 `fact_sales` 的 `sales_date` 衍生無法保證連續性。

**Import 時的額外優化建議：**
- `fact_sales` 中的 `sales_date` 確保是 `DATE` 型別（非 timestamp），匯入後欄位會是整數型 date key，VertiPaq 壓縮率最高
- 如果記憶體有壓力，可在 PostgreSQL 建一個 View 只取 Power BI 需要的欄位，避免匯入不必要的欄位

***

## 2. Power BI 資料模型關聯設計

這是你的 Star Schema 在 Power BI 中的完整 Relationship 配置：

| From Table (Many) | From Column | To Table (One) | To Column | Cardinality | Direction |
|---|---|---|---|---|---|
| `fact_sales` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single (→) |
| `fact_sales` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single (→) |
| `fact_sales` | `vendor_sk` | `dim_vendor` | `vendor_sk` | Many-to-One | Single (→) |
| `fact_sales` | `sales_date` | `dim_date` | `date_key` | Many-to-One | Single (→) |
| `fact_inventory_snapshot` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single (→) |
| `fact_inventory_snapshot` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single (→) |
| `fact_inventory_snapshot` | `snapshot_date` | `dim_date` | `date_key` | Many-to-One | Single (→) |

**關鍵設計決策：**

- **兩張 Fact Table 都連接同一張 `dim_date`**，這正是你使用 `dim_date` 的核心理由——它是兩張事實表的 **Role-Playing Dimension** 橋梁
- **Cross-filter direction 全部設 Single（單向）**，避免 `dim_product` 的篩選意外透過 `fact_sales` 影響 `fact_inventory_snapshot`，這是標準 Star Schema 最佳實踐
- `dim_date` 建立後，在 Power BI 右鍵標記為 **"Mark as Date Table"**，否則 Time Intelligence DAX 函數不會生效

***

## 3. 核心 DAX — Inventory Turnover & DSI

你的 `fact_inventory_snapshot` 有 `BEGINNING` 與 `ENDING` 兩類快照，這是計算**平均庫存**的黃金資料，公式如下：


$$
\text{Inventory Turnover} 
= \frac{\text{COGS (期間銷售成本)}}{\text{Average Inventory}} 
$$


$$
    \text{DSI} = \frac{365}{\text{Inventory Turnover}} 
$$

### Step 1：基礎 Measure（建議建在獨立 `_Measures` 表中）



In [ ]:

-- 1. 期間總銷售成本（來自 fact_sales）
Total COGS =
SUMX(
    fact_sales,
    fact_sales[estimated_cogs]
)

-- 2. 期初庫存總值（BEGINNING 快照）
Beginning Inventory Value =
CALCULATE(
    SUM(fact_inventory_snapshot[total_inventory_value]),
    fact_inventory_snapshot[snapshot_type] = "BEGINNING"
)

-- 3. 期末庫存總值（ENDING 快照）
Ending Inventory Value =
CALCULATE(
    SUM(fact_inventory_snapshot[total_inventory_value]),
    fact_inventory_snapshot[snapshot_type] = "ENDING"
)

-- 4. 平均庫存（期初+期末 / 2）
Average Inventory Value =
DIVIDE(
    [Beginning Inventory Value] + [Ending Inventory Value],
    2
)



### Step 2：核心 KPI Measures


In [ ]:
-- 庫存周轉率 (Inventory Turnover)
Inventory Turnover =
DIVIDE(
    [Total COGS],
    [Average Inventory Value],
    BLANK()   -- 分母為 0 時回傳 BLANK 而非錯誤，避免 visual 顯示 Infinity
)

-- 庫存銷售天數 (Days Sales of Inventory)
-- 注意：分母用 Inventory Turnover，避免重複計算
Days Sales of Inventory (DSI) =
VAR _turnover = [Inventory Turnover]
RETURN
    IF(
        _turnover = 0 || ISBLANK(_turnover),
        BLANK(),
        DIVIDE(365, _turnover)
    )


### Step 3：Period-Aware 版本（支援 YTD 篩選）

當使用者在報表中選擇「年份」或「月份」時，上面的 Measures 已自動跟隨 `dim_date` 的篩選上下文正確計算。若你需要固定計算**全年度**對比，可加一個 YTD 版本：


In [ ]:
Inventory Turnover YTD =
CALCULATE(
    [Inventory Turnover],
    DATESYTD(dim_date[date_key])
)


***

## 

Phase 5 後續還需要的 DAX：
- **缺貨率**：需要用 `fact_inventory_snapshot` 中 `quantity_on_hand = 0` 的快照比例計算
- **呆滯庫存**：需定義「超過 N 天未銷售」的商品邏輯，建議用 `EXCEPT` 或 `DATESBETWEEN` 配合 `fact_sales` 做排除
- **ABC 分類**：用 `RANKX` 對品牌/商品排名，劃分 A/B/C 貢獻層



對於擁有 1500 萬筆資料的數據集，我們在設計 DAX 時必須非常注意 **VertiPaq 引擎的效能**。

***

### 1. 缺貨率 (Stockout Rate)

**業務定義**：在特定的時間段、門店或商品範圍內，庫存數量為 0 的比例。
**資料來源**：`fact_inventory_snapshot`。由於我們有快照資料，最精準的做法是計算「總快照紀錄」中「庫存量 ≤ 0 的快照紀錄」之佔比。


In [ ]:
-- 1. 總庫存快照紀錄數
Total Snapshot Records = 
COUNTROWS('fact_inventory_snapshot')

-- 2. 缺貨快照紀錄數 (庫存為 0)
Out of Stock Records = 
CALCULATE(
    [Total Snapshot Records],
    'fact_inventory_snapshot'[quantity_on_hand] <= 0
)

-- 3. 缺貨率 (Stockout Rate)
Stockout Rate = 
DIVIDE([Out of Stock Records], [Total Snapshot Records], BLANK())


> **💡 這組 DAX 非常輕量，能讓使用者在報表上任意透過 `dim_store` (店鋪)、`dim_product` (品牌/商品) 或 `dim_date` (時間) 進行下鑽分析。如果某店鋪特定月份的 Stockout Rate 異常飆高，就能立即被看出來。

***

### 2. 呆滯庫存分析 (Dead Stock Analysis)

**業務定義**：目前倉庫有庫存（Quantity on Hand > 0），但在過去 N 天（通常設為 90 天或 180 天）內**完全沒有銷售紀錄**的商品。這會積壓資金。
**挑戰**：需要同時跨越兩張 Fact Tables (`fact_inventory_snapshot` 看當前庫存，`fact_sales` 看歷史銷售)。

這裡我們實作一個業界常用的 **「過去 90 天無銷售之呆滯庫存」** 邏輯：


In [ ]:


-- 1. 過去 90 天的總銷售量
Sales Qty Last 90 Days = 
CALCULATE(
    SUM('fact_sales'[sales_quantity]),
    DATESINPERIOD(
        'dim_date'[date_key], 
        MAX('dim_date'[date_key]), -- 基準點：當前篩選上下文的最後一天
        -90, 
        DAY
    )
)

-- 2. 呆滯庫存價值 (Dead Stock Value)
-- 邏輯：只挑選「期末有庫存」且「過去90天銷售量為0或空值」的商品，加總其庫存價值
Dead Stock Value (90 Days) = 
CALCULATE(
    [Ending Inventory Value],  -- 來自我們 Part 1 寫的 Measure
    FILTER(
        'dim_product',
        [Sales Qty Last 90 Days] = 0 || ISBLANK([Sales Qty Last 90 Days])
    ),
    'fact_inventory_snapshot'[quantity_on_hand] > 0
)




> **💡 優化建議**：`FILTER('dim_product', ...)` 是迭代函數。因為我們在 PostgreSQL 階段已經把維度表設計得很乾淨，對 `dim_product` 迭代的效能會比直接對百萬筆的 Fact 表迭代好很多。

***

### 3. ABC 分類 (ABC Classification / Pareto Analysis)

**業務定義**：根據「帕累托法則 (80/20法則)」，將商品依據「貢獻的銷售額」排序：
* **A 類**：貢獻前 70% 營收的商品（核心商品，絕不能缺貨）
* **B 類**：貢獻接下來 20% 營收的商品
* **C 類**：貢獻最後 10% 營收的商品（可能是呆滯庫存的高風險群）

**架構決策**：
針對 1200 萬筆級別的資料，動態 ABC 分類（在 DAX 裡算 Running Total）非常消耗 CPU 資源（可能導致 "Resources Exceeded" 錯誤）。 **在作品集中採用「雙軌制」** 以展現架構思維：

1. **靜態 ABC (企業級資料倉儲)**：在 dbt/PostgreSQL 的 ELT 流程中，基於去年的年度總銷售額，算好 ABC 等級寫死在 `dim_product` 中作為一個欄位。
2. **動態 ABC (靈活，適合小規模倉儲)**：寫一個 DAX Measure 來應對「使用者想任意選取特定月份/特定店鋪」的情境。以下是動態版 DAX：


In [ ]:
-- 1. 總營收
Total Revenue = SUMX('fact_sales', 'fact_sales'[sales_dollars])

-- 2. 動態 ABC 分類 Measure (放進矩陣/表格 visual 中使用)
Dynamic ABC Class = 
VAR CurrentProductRevenue = [Total Revenue]

RETURN  -- 👈 修正：必須加上這個 RETURN 才能開始 IF 判斷
    IF(
        ISBLANK(CurrentProductRevenue) || CurrentProductRevenue = 0,
        BLANK(),
        
        -- 只有當營收 > 0 時，才宣告內部變數並計算 Running Total
        VAR TotalRevenueAllProducts = CALCULATE([Total Revenue], ALL('dim_product'))
        
        VAR CumulativeRevenue = 
            CALCULATE(
                [Total Revenue],
                FILTER(
                    ALL('dim_product'),
                    [Total Revenue] >= CurrentProductRevenue
                )
            )
            
        VAR CumulativePct = DIVIDE(CumulativeRevenue, TotalRevenueAllProducts)
        
        RETURN  -- 👈 內部變數區塊也需要自己的 RETURN
            SWITCH(
                TRUE(),
                CumulativePct <= 0.70, "A",
                CumulativePct <= 0.90, "B",
                "C"
            )
    )

***

## 靜態 ABC



###  `dim_product` 新增欄位

在建立 `dim_product` 的 `SELECT` 清單最後，加入了 `abc_class` 欄位，以 `NULL::VARCHAR(1)` 作為占位符：

```sql
-- ABC placeholder: populated after fact_sales load (see Step 6)
NULL::VARCHAR(1)  AS abc_class
```

原因是 `dim_product` 必須在 `fact_sales` **之前**建立（fact 表需要 `product_sk` 做 JOIN），所以此時還無法計算 ABC — 這就是「兩階段」策略的核心。

### `UPDATE ... FROM CTE` 回寫 ABC 標籤

在 `fact_inventory_snapshot` 之後，新增了完整的靜態 ABC 計算塊，分四個 Phase：

| Phase | 做什麼 |
|---|---|
| **A** `product_revenue` | 從 `fact_sales` 聚合每個 `product_sk` 的 `SUM(sales_dollars)` |
| **B** `running_total` | 用 `SUM() OVER(ORDER BY total_sales_dollars DESC ROWS BETWEEN ...)` 計算累積百分比 `cumulative_pct` |
| **C** `abc_labels` | 用 `CASE WHEN` 套入閾值：`<= 80` → A，`<= 95` → B，其餘 → C |
| **D** `UPDATE` | 一次性 `UPDATE marts.dim_product SET abc_class = al.abc_class FROM abc_labels al WHERE product_sk = product_sk` |

### Step 7 — 驗證查詢（已 comment out）

附上一段可手動執行的驗證 SQL，確認 Pareto 分佈是否合理（A 約佔 10–20% SKU，C 約佔 50–70% SKU）。

***

## 為何這樣設計

這個設計解決了一個典型的 **Fact/Dim 循環依賴問題**：

- `dim_product` 要先建立，因為 `fact_sales` 需要 `JOIN` 它取得 `product_sk`。
- 但 ABC 分類需要 `fact_sales` 的銷售數字才能計算。
- **解法**：把 `abc_class` 欄位設為 `NULL` 佔位，等 `fact_sales` 建好後，再用一次 `UPDATE ... FROM CTE` 反寫回去。這是一個乾淨的兩階段方法，不需要暫存表、不需要重新 `DROP`/`CREATE dim_product`，對 12M 筆 fact 表也只需掃描一次聚合。

***




***

## 補充 Dead Stock（呆滯庫存）



In [ ]:

-- ================================================================
-- DEAD STOCK ANALYSIS  (Group: 04_Dead Stock)
-- ================================================================



-- [輔助] 呆滯商品數（SKU 計數）
Dead Stock SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        -- 條件 A：過去 90 天無銷售
        ( [Sales Qty Last 90 Days] = 0 || ISBLANK([Sales Qty Last 90 Days]) )
        &&
        -- 條件 B：期末仍有庫存（真正的積壓）
        CALCULATE(
            SUM('fact_inventory_snapshot'[quantity_on_hand]),
            'fact_inventory_snapshot'[snapshot_type] = "ENDING"
        ) > 0
    )
)


-- [核心 KPI] 呆滯庫存佔比（佔總期末庫存金額的百分比）
Dead Stock % of Total Inventory =
DIVIDE(
    [Dead Stock Value (90 Days)],
    [Ending Inventory Value],
    BLANK()
)


***

## Reorder Point（再訂購點）

這是本次的核心新增 KPI。公式為：

$$
\text{Reorder Point} = (\text{Average Daily Sales}) \times \text{Lead Time (Days)} + \text{Safety Stock}
$$

由於原始資料集中沒有 Lead Time 欄位，採用業界常見的**統計安全庫存法**，以銷售標準差估算：


In [ ]:

-- ================================================================
-- REORDER POINT  (Group: 05_Reorder Point)
-- ================================================================

-- [輔助] 日均銷售量（基於 dim_date 的實際有銷售日）
Avg Daily Sales Qty =
VAR _totalQty = SUM('fact_sales'[sales_quantity])
VAR _activeDays =
    CALCULATE(
        DISTINCTCOUNT('dim_date'[date_key]),
        CROSSFILTER('fact_sales'[sales_date], 'dim_date'[date_key], BOTH)
    )
RETURN
    DIVIDE(_totalQty, _activeDays, BLANK())

-- [輔助] 銷售量標準差（用於計算安全庫存，衡量需求波動）
Stddev Daily Sales Qty =
-- DAX 沒有內建 STDEV over dates，用 SUMX 模擬日銷售量後計算
VAR _salesByDay =
    ADDCOLUMNS(
        VALUES('dim_date'[date_key]),
        "@DailySales",
        CALCULATE(SUM('fact_sales'[sales_quantity]))
    )
VAR _avgDaily = AVERAGEX(_salesByDay, [@DailySales])
VAR _variance =
    AVERAGEX(
        _salesByDay,
        ( [@DailySales] - _avgDaily ) ^ 2
    )
RETURN
    SQRT(_variance)

-- [輔助] 安全庫存量
-- 公式：Z-score (95% 服務水準 = 1.65) × Stddev × SQRT(Lead Time)
-- Lead Time 預設 7 天（可調整為參數表）
Safety Stock Qty =
VAR _leadTimeDays = 7          -- 假設補貨前置天數，可改為 SELECTEDVALUE(参数表[LeadTime])
VAR _zScore = 1.65             -- 95% 服務水準對應的 Z 值
RETURN
    _zScore
    * [Stddev Daily Sales Qty]
    * SQRT(_leadTimeDays)

-- [核心 KPI] 再訂購點（數量）
Reorder Point Qty =
VAR _leadTimeDays = 7
VAR _demandDuringLead = [Avg Daily Sales Qty] * _leadTimeDays
RETURN
    ROUND(
        _demandDuringLead + [Safety Stock Qty],
        0    -- 庫存量通常取整數
    )

-- [核心 KPI] 再訂購點提示（結合當前庫存，判斷是否需要立即補貨）
Reorder Alert =
VAR _currentStock =
    CALCULATE(
        SUM('fact_inventory_snapshot'[quantity_on_hand]),
        'fact_inventory_snapshot'[snapshot_type] = "ENDING"
    )
VAR _rop = [Reorder Point Qty]
RETURN
    SWITCH(
        TRUE(),
        ISBLANK(_rop) || ISBLANK(_currentStock), BLANK(),
        _currentStock <= 0,               "🔴 Out of Stock",
        _currentStock <= _rop,            "🟡 Reorder Now",
        _currentStock <= _rop * 1.2,      "🟠 Low Stock",
        "🟢 OK"
    )




***

## Power BI 報表使用建議

| Measure | 建議 Visual | 說明 |
|---|---|---|
| `Dead Stock Value (90 Days)` | Card / KPI | 頂層 Summary |
| `Dead Stock % of Total Inventory` | Gauge | 呆滯佔比，目標線設 5% |
| `Dead Stock SKU Count` | Card | 呆滯商品數量 |
| `Reorder Point Qty` | Matrix（按 Product / Store）| 逐品項逐店顯示 |
| `Reorder Alert` | Table with Conditional Formatting | 顏色標記，按 ABC Class 篩選 A 類先看 |
| `Avg Daily Sales Qty` | Tooltip Measure | 放入 Tooltip，輔助 Reorder Point 解釋 |

> **Portfolio 亮點說明**：Reorder Point 採用統計安全庫存法（Z-score × σ × √LT），而非簡單的固定倍數，能體現對供應鏈分析方法論的理解。Lead Time 預設 7 天（可調整為參數表）

## 儀表板頁面

 3 頁 Power BI 報表結構，讓 KPI 有清楚的故事線。 

- Page 1：Executive Overview，放 `Inventory Turnover`、`DSI`、`Stockout Rate`、`Dead Stock Value`、`Dead Stock %` 這些總覽 KPI。
- Page 2：SKU / Store Drilldown，放 `ABC Class`、`Reorder Point Qty`、`Reorder Alert`、`Ending Inventory Value`，用 Matrix + Conditional Formatting 看哪些 A 類商品快要缺貨。 
- Page 3：Inventory Risk Analysis，聚焦 Dead Stock 與 Stockout 的兩端風險，讓面試官一眼看到你不只會做報表，還懂庫存管理邏輯。

## 視覺設計

- Card：`Dead Stock Value (90 Days)`、`Dead Stock SKU Count`、`Inventory Turnover`。
- Matrix：`Product` / `Store` / `abc_class` / `Reorder Point Qty` / `Current Stock` / `Reorder Alert`。
- Bar chart：按 `abc_class` 比較 Dead Stock Value，強化「C 類高呆滯風險、A 類高缺貨風險」的商業洞察。
- Line chart：月度 `DSI`、`Stockout Rate` 趨勢，讓時間分析能力更完整。 


## 頁面架構

整體版面請統一用相同掃描邏輯：最上方 KPI cards、中間分析圖、底部明細或矩陣，並把常用 slicers 維持一致，避免使用者跨頁切換時失去上下文。 


- 全域 slicers：`Year`、`Month`、`Store`、`Vendor`、`ABC Class`。
- 導航按鈕：`1 Overview`、`2 Inventory Risk`、`3 Replenishment`，每頁右上角加 `Last Refresh Date` 與 `Metric Info` tooltip，提升資料信任感。
- 色彩規則：正常用中性色，風險只用少量強調色，例如缺貨紅、低庫存橙、健康綠，避免整頁彩色噪音。 

## Page 1 Overview

這頁回答的問題是：**目前整體庫存績效好不好，哪個方向最值得先追。** KPI 要放在最上方，因為儀表板應先讓主管看到總體表現，再往下看驅動因素。 

### Wireframe
```text
┌──────────────────────────────────────────────────────────────────────┐
│ Title: Inventory Performance Overview                               │
│ Slicers: Year | Month | Store | Vendor | ABC Class                 │
├──────────────────────────────────────────────────────────────────────┤
│ Card 1      │ Card 2      │ Card 3         │ Card 4                │
│ Turnover    │ DSI         │ Stockout Rate  │ Dead Stock %          │
├──────────────────────────────────────────────────────────────────────┤
│ Line chart: Monthly Turnover & DSI Trend                            │
├───────────────────────────────────────┬──────────────────────────────┤
│ Column chart: Stockout Rate by Store  │ Bar chart: Dead Stock by ABC│
├───────────────────────────────────────┴──────────────────────────────┤
│ Matrix: Store / ABC / Revenue / Ending Inv / DSI / Stockout Rate    │
└──────────────────────────────────────────────────────────────────────┘
```

### Visuals / Measures
- KPI Cards  
  - `Inventory Turnover`
  - `Days Sales of Inventory (DSI)`
  - `Stockout Rate`
  - `Dead Stock % of Total Inventory`。
- Line chart  
  - Axis: `dim_date[month_year]`
  - Values: `Inventory Turnover`, `DSI`，雙軸或切換按鈕均可。
- Column chart  
  - Axis: `dim_store[store_name]`
  - Value: `Stockout Rate`，依高到低排序，快速抓出風險門店。
- Bar chart  
  - Axis: `dim_product[abc_class]`
  - Value: `Dead Stock Value (90 Days)`，突出 C 類積壓是否過高。
- Bottom matrix  
  - Rows: `Store`, `ABC Class`
  - Values: `Total Revenue`, `Ending Inventory Value`, `Inventory Turnover`, `DSI`, `Stockout Rate`。

### 設計重點
這頁不要放太多表格，重點是先讓人看出「周轉慢」還是「缺貨多」還是「呆滯高」。若你要做作品集，這頁最好能一句話對應一個管理問題，例如「A 類缺貨風險」與「C 類資金占壓」分開呈現。 

## Page 2 Inventory Risk

這頁回答的問題是：**哪些商品或門店同時面臨缺貨與呆滯的雙重風險。** 庫存儀表板通常需要明確的 alerts 區塊與風險分群，這也是實務上最有操作價值的頁面。 

### Wireframe
```text
┌──────────────────────────────────────────────────────────────────────┐
│ Title: Inventory Risk Analysis                                      │
│ Slicers: Year | Month | Store | Vendor | ABC Class | Product        │
├──────────────────────────────────────────────────────────────────────┤
│ Card 1            │ Card 2              │ Card 3                    │
│ Dead Stock Value  │ Dead Stock SKU Cnt  │ Out of Stock Records      │
├───────────────────────────────────────┬──────────────────────────────┤
│ Scatter: Stockout vs Dead Stock       │ Treemap/Bar: Risk by Store  │
├───────────────────────────────────────┴──────────────────────────────┤
│ Table: Product / Store / ABC / QOH / Sales 90D / Dead? / Risk Flag  │
└──────────────────────────────────────────────────────────────────────┘
```

### Visuals / Measures
- KPI Cards  
  - `Dead Stock Value (90 Days)`
  - `Dead Stock SKU Count`
  - `Out of Stock Records` 或 `Stockout Rate`。
- Scatter chart  
  - X: `Stockout Rate`
  - Y: `Dead Stock % of Total Inventory`
  - Details: `Store` 或 `Product Category`
  - Size: `Ending Inventory Value`，用來看「高庫存但也高風險」的象限。
- Bar / Treemap  
  - Category: `Store` 或 `Vendor`
  - Value: `Dead Stock Value (90 Days)`，快速找風險集中區。
- Detail table  
  - `Product`
  - `Store`
  - `ABC Class`
  - `Ending Quantity On Hand`
  - `Sales Qty Last 90 Days`
  - `Dead Stock Value (90 Days)`
  - 可加一個 `Dead Stock Flag` measure：Yes / No，方便條件格式。

### 建議新增輔助 Measure
若你要讓這頁更像企業報表，可以多做一個簡單旗標。

```dax
Dead Stock Flag =
IF ( [Dead Stock Value (90 Days)] > 0, "Dead Stock", "Active" )
```

### 設計重點
這頁的核心不是趨勢，而是**風險定位**。把條件格式做好，例如 Dead Stock 紅色、正常灰色，使用者會更快找到異常商品。 

## Page 3 Replenishment

這頁回答的問題是：**哪些 SKU 現在該補貨，優先順序是什麼。** 由於補貨判斷需要結合銷售速度、Lead Time、安全庫存與當前庫存，所以最適合放在獨立頁，並以矩陣作為主要操作視圖。 

### Wireframe
```text
┌──────────────────────────────────────────────────────────────────────┐
│ Title: Replenishment & ABC Prioritization                           │
│ Slicers: Year | Month | Store | Vendor | ABC Class | Product        │
├──────────────────────────────────────────────────────────────────────┤
│ Card 1            │ Card 2            │ Card 3                      │
│ Reorder SKU Count │ Avg Daily Sales   │ Avg Ending Inventory Value  │
├───────────────────────────────────────┬──────────────────────────────┤
│ Bar: Reorder Alert Count by ABC       │ Bar: Top Reorder Products   │
├───────────────────────────────────────┴──────────────────────────────┤
│ Matrix: Product / Store / ABC / QOH / ROP / Safety / Alert / Vendor │
└──────────────────────────────────────────────────────────────────────┘
```

### Visuals / Measures
- KPI Cards  
  - `Reorder SKU Count`，建議新增 measure。
  - `Avg Daily Sales Qty`
  - `Ending Inventory Value` 或 `Average Inventory Value`。
- Bar chart 1  
  - Axis: `abc_class`
  - Value: 補貨警示商品數，讓面試官看到你把 ABC 與 replenishment 結合，而不是各做各的。
- Bar chart 2  
  - Axis: `Product`
  - Value: `Reorder Point Qty`
  - Visual level filter: `Reorder Alert = "🟡 Reorder Now"`，聚焦真正需要行動的 SKU。
- Main matrix  
  - Rows: `Product`, `Store`
  - Columns/Values:
    - `ABC Class`
    - `Ending Quantity On Hand`
    - `Avg Daily Sales Qty`
    - `Safety Stock Qty`
    - `Reorder Point Qty`
    - `Reorder Alert`
    - `Vendor`。

### 建議新增輔助 Measures
這兩個 Measure 很適合這頁面。

```dax
Current Stock Qty =
CALCULATE(
    SUM('fact_inventory_snapshot'[quantity_on_hand]),
    'fact_inventory_snapshot'[snapshot_type] = "ENDING"
)

Reorder SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        [Reorder Alert] = "🟡 Reorder Now"
    )
)
```

### 設計重點
這頁要讓人一看就知道「哪些 SKU 該下單」，所以矩陣是主角，圖表只是排序與摘要。再訂購點本來就依賴銷售速度、Lead Time 與安全庫存來計算，因此把 `Avg Daily Sales Qty`、`Safety Stock Qty`、`Reorder Point Qty` 放在同一張表最有說服力。

## Slicers 與互動

Slicers 不要太多，5 到 6 個已經足夠，過量篩選會增加認知負擔；這也符合 Power BI 儀表板常見的設計建議。 

建議固定如下：
- `Year`
- `Month`
- `Store`
- `Vendor`
- `ABC Class`
- 第 2、3 頁可額外加 `Product` 搜尋式 slicer。 

互動設定建議：
- Overview 的圖表可 cross-highlight，但底部 matrix 建議只接受篩選，不反向影響上方 KPI，避免使用者看亂。
- 第 1 頁點擊 `Store` 後 drill-through 到第 2 頁；點擊 `Product` 後 drill-through 到第 3 頁，這種 progressive disclosure 很適合多頁報表。

## 作品集呈現

- Page 1：How healthy is inventory performance overall? 
- Page 2：Where are stockout and dead stock risks concentrated? 
- Page 3：Which SKUs should be reordered first, considering ABC priority? 

下一步最適合直接做的是：**我幫你把這 3 頁 wireframe 進一步細化成 Power BI 可直接照著擺的「版面座標圖 + visual titles + tooltip 設計」**。

## 問題解決

### ABC 有空值

沒有銷售紀錄但有庫存的 SKU 很自然會落成空值。
這類 SKU 在業務上不一定等於 C 類；它們可能是：

 - 新品，還沒開始賣。

 - 長尾品，全年沒賣但仍有庫存。

 - 主檔或對應不完整，導致沒被分類。

 - 只存在庫存快照、沒有進入銷售聚合邏輯的商品。

 - 如果你把這些全部強制歸 C，就等於把「分類邏輯未覆蓋」誤當成「低貢獻商品」，這在方法論上不夠乾淨。


### 1) 顯示用欄位


In [ ]:

ABC Class Display =
COALESCE ( 'dim_product'[abc_class], "Unclassified" )
```
這個欄位專門給 axis、rows、legend 用，直接把空值變成 `Unclassified`，避免圖表出現 `(Blank)`。 [ppl-ai-file-upload.s3.amazonaws](https://ppl-ai-file-upload.s3.amazonaws.com/web/direct-files/attachments/82717827/25964d68-9551-404f-a43d-17fc8303bb14/data02-2.csv)

### 2) 排序欄位
```dax
ABC Class Sort =
SWITCH(
    TRUE(),
    'dim_product'[abc_class] = "A", 1,
    'dim_product'[abc_class] = "B", 2,
    'dim_product'[abc_class] = "C", 3,
    4
)


建立後在 Power BI 把 `ABC Class Display` 設定成 **Sort by column = `ABC Class Sort`**，這樣排序會固定為 A → B → C → Unclassified，不會跳字母順序。



### 3) 優先群組欄位


In [ ]:
ABC Priority Group =
SWITCH(
    TRUE(),
    'dim_product'[abc_class] = "A", "A - Critical",
    'dim_product'[abc_class] = "B", "B - Important",
    'dim_product'[abc_class] = "C", "C - Long Tail",
    "Review - Unclassified"
)


這個欄位適合放在補貨頁或 table tooltip，讓業務語言比單純 A/B/C 更清楚。

### 4) 未分類 SKU 數量


In [ ]:

Unclassified SKU Count =
CALCULATE(
    DISTINCTCOUNT('dim_product'[product_sk]),
    FILTER(
        'dim_product',
        ISBLANK('dim_product'[abc_class])
    )
)


這個 measure 放在 Page 2 或 tooltip，告訴使用者目前有多少 SKU 未進入 ABC 邏輯。 

### 5) 未分類庫存金額


In [ ]:
Unclassified Inventory Value =
CALCULATE(
    [Ending Inventory Value],
    FILTER(
        'dim_product',
        ISBLANK('dim_product'[abc_class])
    )
)


這個 measure 很適合做 Card 或 tooltip，因為目前已經知道部分 store 的未分類庫存值不小。 

### 6) 未分類呆滯庫存金額


In [ ]:
Unclassified Dead Stock Value =
CALCULATE(
    [Dead Stock Value (90 Days)],
    FILTER(
        'dim_product',
        ISBLANK('dim_product'[abc_class])
    )
)


這個 measure 能回答未分類商品到底占了多少 dead stock。 
### 7) 分類品質旗標


ABC Classification Status =
IF(
    ISBLANK(SELECTEDVALUE('dim_product'[abc_class])),
    "Unclassified",
    "Classified"
)


這個 measure 適合放在明細表或 tooltip，不建議做主軸，但很適合 drillthrough 頁面。

## 圖表標題

Power BI 支援 expression-based title / subtitle，所以建議你把高價值視覺做成動態標題，讓使用者知道目前看的篩選上下文。

### 1) Dead Stock by ABC 主圖標題


In [ ]:
Title - Dead Stock by ABC =
VAR _store =
    SELECTEDVALUE('dim_store'[store_number], "All Stores")
VAR _year =
    SELECTEDVALUE('dim_date'[year], "All Years")
RETURN
    "Dead Stock Value (90 Days) by ABC Class | Store: " & _store & " | Year: " & _year


這種標題符合 chart title 最佳實踐：直接說明圖表內容與篩選條件，不讓使用者猜現在看到的是哪個 store 或哪一年。 




## README 內容

完成報表後，下一步就是補 README，因為這個專案本來就定位成 BI Analyst / Data Engineer 作品集，而你也一直在強調 GitHub 呈現品質。

README 建議新增 4 個段落：
- `Business Problem`：企業如何在高銷售與高庫存成本之間平衡。
- `Data Architecture`：Raw → Staging → Marts、Star Schema、靜態 ABC 回寫策略。 
- `KPI Logic`：逐條說明 Inventory Turnover、DSI、Stockout Rate、Dead Stock、Reorder Point 的商業定義與公式假設。 
- `Portfolio Highlights`：特別寫出「12M+ fact table 下，將 ABC 改為 PostgreSQL 靜態預計算，避免 Power BI 動態 Running Total 資源爆掉」這個亮點。 

